# Real Estate Price Prediction
**Dataset:** Bengaluru House Price Data (Kaggle)  
**Author:** Naveen-Boddepalli  
**Event:** GSSoC 2026

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import joblib

# DL
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("All imports successful!")

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../Dataset/Bengaluru_House_Data.csv')
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())

## 3. Data Preprocessing

In [ ]:
# Drop rows where location or size is null (very few)
df.dropna(subset=['location', 'size'], inplace=True)

# Fill missing bath with median
df['bath'] = df['bath'].fillna(df['bath'].median())
df['balcony'] = df['balcony'].fillna(df['balcony'].median())

# Drop society (too many nulls, low signal)
df.drop(columns=['society'], inplace=True)

print("After basic cleaning:", df.shape)

In [ ]:
# Parse 'size' -> BHK count
def parse_bhk(size):
    try:
        return int(str(size).split()[0])
    except:
        return np.nan

df['bhk'] = df['size'].apply(parse_bhk)
df.dropna(subset=['bhk'], inplace=True)
df['bhk'] = df['bhk'].astype(int)
df.drop(columns=['size'], inplace=True)
print(df['bhk'].value_counts().head(10))

In [ ]:
# Parse 'total_sqft' (handles ranges like "2000-2500")
def parse_sqft(sqft):
    try:
        if '-' in str(sqft):
            parts = sqft.split('-')
            return (float(parts[0]) + float(parts[1])) / 2
        return float(sqft)
    except:
        return np.nan

df['total_sqft'] = df['total_sqft'].apply(parse_sqft)
df.dropna(subset=['total_sqft'], inplace=True)
print(f"Shape after sqft parsing: {df.shape}")

In [ ]:
# Engineer price_per_sqft
df['price_per_sqft'] = df['price'] * 1e5 / df['total_sqft']

# Group rare locations
location_counts = df['location'].value_counts()
df['location'] = df['location'].apply(
    lambda x: x.strip() if location_counts[x.strip()] >= 10 else 'other'
)
print(f"Unique locations after grouping: {df['location'].nunique()}")

In [ ]:
# Remove outliers: price_per_sqft outside mean +/- std per location
def remove_pps_outliers(df):
    df_out = pd.DataFrame()
    for loc, subdf in df.groupby('location'):
        m = subdf['price_per_sqft'].mean()
        s = subdf['price_per_sqft'].std()
        reduced = subdf[(subdf['price_per_sqft'] > (m - s)) & (subdf['price_per_sqft'] < (m + s))]
        df_out = pd.concat([df_out, reduced], ignore_index=True)
    return df_out

df = remove_pps_outliers(df)

# Remove extreme bath counts (bath > bhk+2 is suspicious)
df = df[df['bath'] <= df['bhk'] + 2]

print(f"Shape after outlier removal: {df.shape}")

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Price distribution
axes[0].hist(df['price'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution (Lakhs)', fontsize=13)
axes[0].set_xlabel('Price'); axes[0].set_ylabel('Count')

# Sqft vs Price scatter
axes[1].scatter(df['total_sqft'], df['price'], alpha=0.3, color='coral', s=5)
axes[1].set_title('Total Sqft vs Price', fontsize=13)
axes[1].set_xlabel('Total Sqft'); axes[1].set_ylabel('Price (Lakhs)')

# BHK distribution
df['bhk'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='mediumseagreen')
axes[2].set_title('BHK Distribution', fontsize=13)
axes[2].set_xlabel('BHK'); axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../Images/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 6))
numeric_cols = ['total_sqft', 'bath', 'balcony', 'bhk', 'price_per_sqft', 'price']
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('../Images/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 10 locations by median price
top10 = df.groupby('location')['price'].median().sort_values(ascending=False).head(10)
plt.figure(figsize=(12, 5))
top10.plot(kind='bar', color='orchid', edgecolor='white')
plt.title('Top 10 Locations by Median Price (Lakhs)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Median Price (Lakhs)')
plt.tight_layout()
plt.savefig('../Images/top_locations.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Engineering & Train/Test Split

In [ ]:
# Drop derived feature before modeling (avoid leakage)
df.drop(columns=['price_per_sqft'], inplace=True)

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Preprocessing pipeline
categorical_features = ['area_type', 'availability', 'location']
numerical_features = ['total_sqft', 'bath', 'balcony', 'bhk']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

## 6. ML Models

In [ ]:
results = {}

def evaluate(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    results[name] = {'R2': round(r2, 4), 'MAE': round(mae, 2), 'RMSE': round(rmse, 2)}
    print(f"{name}: R2={r2:.4f} | MAE={mae:.2f} | RMSE={rmse:.2f}")

# Linear Regression
lr_pipe = Pipeline([('pre', preprocessor), ('model', LinearRegression())])
lr_pipe.fit(X_train, y_train)
evaluate('Linear Regression', y_test, lr_pipe.predict(X_test))

# Lasso
lasso_pipe = Pipeline([('pre', preprocessor), ('model', Lasso(alpha=1.0))])
lasso_pipe.fit(X_train, y_train)
evaluate('Lasso', y_test, lasso_pipe.predict(X_test))

# Random Forest
rf_pipe = Pipeline([('pre', preprocessor), ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])
rf_pipe.fit(X_train, y_train)
evaluate('Random Forest', y_test, rf_pipe.predict(X_test))

# XGBoost
xgb_pipe = Pipeline([('pre', preprocessor), ('model', XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1))])
xgb_pipe.fit(X_train, y_train)
evaluate('XGBoost', y_test, xgb_pipe.predict(X_test))

In [ ]:
# Save best ML model
best_ml_model = xgb_pipe  # Update if another model wins
joblib.dump(best_ml_model, 'real_estate_best_model.pkl')
print("Saved: real_estate_best_model.pkl")

## 7. Deep Learning Models (PyTorch)

In [ ]:
# Preprocess for DL: use the same preprocessor fitted on ML data
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

X_train_t = torch.FloatTensor(X_train_proc)
y_train_t = torch.FloatTensor(y_train.values).unsqueeze(1)
X_test_t  = torch.FloatTensor(X_test_proc)
y_test_t  = torch.FloatTensor(y_test.values).unsqueeze(1)

input_dim = X_train_t.shape[1]
print(f"Input dimension: {input_dim}")

train_ds = TensorDataset(X_train_t, y_train_t)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)

In [ ]:
# ── MLP ──────────────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),        nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x)

# ── Wide & Deep ───────────────────────────────────────────────────────
class WideDeep(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.wide = nn.Linear(input_dim, 1)
        self.deep = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.wide(x) + self.deep(x)

def train_model(model, epochs=50, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    loss_fn = nn.MSELoss()
    model.train()
    for epoch in range(epochs):
        total = 0
        for xb, yb in train_dl:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item()
        avg = total / len(train_dl)
        scheduler.step(avg)
        if (epoch+1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs} — loss: {avg:.4f}")
    return model

def eval_dl(name, model):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).squeeze().numpy()
    evaluate(name, y_test.values, preds)

In [ ]:
print("Training MLP...")
mlp = MLP(input_dim)
mlp = train_model(mlp, epochs=50)
eval_dl("MLP", mlp)

print("\nTraining Wide & Deep...")
wd = WideDeep(input_dim)
wd = train_model(wd, epochs=50)
eval_dl("Wide & Deep", wd)

## 8. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
print(results_df.to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric in zip(axes, ['R2', 'MAE', 'RMSE']):
    colors = ['steelblue' if i > 0 else 'coral' for i in range(len(results_df))]
    results_df[metric].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'{metric} by Model', fontsize=13)
    ax.set_xticklabels(results_df.index, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(metric)

plt.tight_layout()
plt.savefig('../Images/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Conclusions

> *(Update after running — best model, key insights, feature importance)*

- **Best model:** TBD
- **Key features:** location, total_sqft, bhk
- **Suggestions:** TabNet and XGBoost likely top performers on this tabular dataset